<!-- ---
title: Deploy VibeVoice ASR on Microsoft Foundry: Long-Form Transcription, +50 Languages Supported & Speaker Diarization
author: Alvaro Bartolome
--- -->

# Deploy NVIDIA Nemotron 3 Nano Omni on Microsoft Foundry: Unifying video, audio, image and text understanding.

![](./thumbnail.png)

In this example, you will deploy [`nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-FP8`](https://huggingface.co/microsoft/nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-FP8) on Microsoft Foundry and then use it to answer questions based on technical documents and meeting recordings.

**NVIDIA Nemotron 3 Nano Omni** is a multimodal large language model that unifies video, audio, image, and text understanding into a single, highly efficient open model. Built to replace fragmented vision‑language‑audio stacks, it extends the Nemotron Nano family with integrated video+speech comprehension, Graphical User Interface (GUI), Optical Character Recognition (OCR), and speech transcription capabilities, enabling enterprise-grade Q&A, summarization, transcription, and document intelligence workflows.

![NVIDIA Nemotron 3 Nano Omni Multimodal Benchmarks](./nemotron-3-nano-omni-benchmarks.png)


The Nemotron 3 Nano Omni architecture extends Nemotron 3 Nano by bringing multimodal perception and reasoning into a single 30B hybrid MoE model.

- **🧠 Hybrid MoE architecture**: Combines Mamba layers for sequence and memory efficiency with transformer layers for precise reasoning.
- **📷 Spatiotemporal visual processing and efficient video sampling**: The inference-time Efficient Video Sampling (EVS) layer compresses the high-density visual tokens from multiple frames into a concise set that the LLM can process without overwhelming its context window.
- **🎨 Multimodal architecture**: 
    - A strong text model at its core, preserving the foundation model’s language ability.
    - Audio integration built upon the NVIDIA Parakeet encoder and specialized datasets that move beyond simple transcription.
    - C-RADIOv4-H and Encoder-based Video Summarization to handle high-resolution images and dynamic video.


![NVIDIA Nemotron 3 Nano Omni Workflow](./nemotron-3-nano-omni-workflow.png)


For more information, make sure to check [the model card on the Hugging Face Hub](https://huggingface.co/nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-FP8), or the official [NVIDIA blog post](https://developer.nvidia.com/blog/nvidia-nemotron-3-nano-omni-powers-multimodal-agent-reasoning-in-a-single-efficient-open-model/).

## Requirements

To run this example, you need to meet the following prerequisites. You can also read more about them in the [Azure Machine Learning Tutorial: Create resources you need to get started](https://learn.microsoft.com/en-us/azure/machine-learning/quickstart-create-resources?view=azureml-api-2).

- You have a Microsoft Azure subscription and are logged in
- You have the `az` CLI installed
- You have the necessary permissions to:
    - Create an Azure Machine Learning Managed Online Endpoint
    - Create an Azure Machine Learning Deployment
- You have Python 3.10+ installed locally and `pip`

For more information, please go through the steps in the guide ["Configure Azure Machine Learning and Microsoft Foundry"](https://huggingface.co/docs/microsoft-azure/guides/configure-azure-ml-microsoft-foundry).

## Setup

### Set environment variables

For convenience, you can set the following environment variables to be used throughout the example:

In [ ]:
%env LOCATION eastus
%env SUBSCRIPTION_ID <YOUR_SUBSCRIPTION_ID>
%env RESOURCE_GROUP <YOUR_RESOURCE_GROUP>
%env WORKSPACE_NAME <YOUR_WORKSPACE_NAME>
%env MODEL_ID nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-FP8

env: LOCATION=eastus
env: SUBSCRIPTION_ID=96f8b384-0587-41d4-9105-9fe6dca745b3
env: RESOURCE_GROUP=huggingface-azure-rg
env: WORKSPACE_NAME=huggingface-azure-project
env: MODEL_ID=nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-FP8


In [14]:
import os
from uuid import uuid4

os.environ["ENDPOINT_NAME"] = f"nemotron-omni-{str(uuid4())[:8]}"
os.environ["DEPLOYMENT_NAME"] = f"nemotron-omni-{str(uuid4())[:8]}"

### Install Azure Python SDK (+ dependencies)

You need to install some Azure Python SDK dependencies:

* `azure-identity` to use the `DefaultAzureCredential` authentication with your Managed Identity
* `azure-ai-ml` to create the Azure Machine Learning Managed Online Endpoint + Deployment, and to invoke it

In [10]:
%pip install azure-ai-ml azure-identity --upgrade --quiet


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Authenticate to Azure Machine Learning

You can then authenticate to Azure Machine Learning with your Managed Identity with Python as:

In [15]:
import os
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

client = MLClient(
    credential=DefaultAzureCredential(),
    subscription_id=os.getenv("SUBSCRIPTION_ID"),
    resource_group_name=os.getenv("RESOURCE_GROUP"),
    workspace_name=os.getenv("WORKSPACE_NAME"),
)

Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


## Create endpoint + deployment

### Create endpoint

Now you need to create the [ManagedOnlineEndpoint via the Azure Machine Learning Python SDK](https://learn.microsoft.com/en-us/python/api/azure-ai-ml/azure.ai.ml.entities.managedonlineendpoint?view=azure-python) as follows:

> [!NOTE]
> Every model in the Hugging Face collection is powered by an efficient inference backend, and each of those can run on a wide variety of instance types (as listed in [Supported Hardware](https://huggingface.co/docs/microsoft-azure/azure-ai/supported-hardware)). Since models and inference engines require a GPU-accelerated instance, you might need to request a quota increase as per [Manage and increase quotas and limits for resources with Azure Machine Learning](https://learn.microsoft.com/en-us/azure/machine-learning/how-to-manage-quotas?view=azureml-api-2).

In [16]:
from azure.ai.ml.entities import ManagedOnlineEndpoint

endpoint = ManagedOnlineEndpoint(name=os.getenv("ENDPOINT_NAME"))
client.begin_create_or_update(endpoint).wait()

### Create deployment

Once the endpoint is created, you need to create the deployment indicating which model, hardware, and settings to use. In this case, we will be using a single NVIDIA H100, enough for deploying our model. Also, we'll be increasing the default timeout per request to 180s to ensure that we can process long image and audio files in time.

In [ ]:
from azure.ai.ml.entities import ManagedOnlineDeployment, OnlineRequestSettings

deployment = ManagedOnlineDeployment(
    name=os.getenv("DEPLOYMENT_NAME"),
    endpoint_name=os.getenv("ENDPOINT_NAME"),
    model=f"azureml://registries/HuggingFace/models/{os.getenv('MODEL_ID').replace('/', '-').replace('_', '-').lower()}/labels/latest",
    instance_type="Standard_NC40ads_H100_v5",
    instance_count=1,
    request_settings=OnlineRequestSettings(request_timeout_ms=180000),
)
client.online_deployments.begin_create_or_update(deployment).result()

![Azure AI Deployment from Azure AI Foundry](./azure-ai-deployment.png)

> [!WARNING]
> The deployment might take ~30 minutes, but it could also take longer depending on the selected SKU availability in the region.

## Run inference on the Foundry Endpoint

Now that the Foundry Endpoint is up and running, we are ready to run inference on it. We will be using the **OpenAI Python SDK** to send requests to our endpoint, which is by served through an **OpenAI-compatible Chat Completions API** at `/v1/chat/completions`.

For this example, we will query the model using all the accepted input modalities (e.g., audio, image, and video). Additionally, we will test the model with a mix of audio and image in the same prompt to see how it performs in multimodality scenarios.

### Set up the OpenAI Python SDK

In [ ]:
%pip install openai --upgrade --quiet

Retrieve the endpoint URL and API key, then create the OpenAI client, making sure to include the `azureml-model-deployment` header so every request is routed to the right deployment. Setting it once via `default_headers` is the recommended approach since the header needs to accompany each request.

In [ ]:
from urllib.parse import urlsplit
from openai import OpenAI
import os

api_key = client.online_endpoints.get_keys(os.getenv("ENDPOINT_NAME")).primary_key
url_parts = urlsplit(client.online_endpoints.get(os.getenv("ENDPOINT_NAME")).scoring_uri)
api_url = f"{url_parts.scheme}://{url_parts.netloc}/v1"

openai_client = OpenAI(
    base_url=api_url,
    api_key=api_key,
    default_headers={"azureml-model-deployment": os.getenv("DEPLOYMENT_NAME")},
)

> [!NOTE]
> Alternatively, you can also build the API URL manually as follows, since the URIs are globally unique per region, meaning that there will only be one endpoint named the same way within the same region:
> ```python
> api_url = f"https://{os.getenv('ENDPOINT_NAME')}.{os.getenv('LOCATION')}.inference.ml.azure.com/v1"
> ```
> Or just retrieve it from either Microsoft Foundry or the Azure Machine Learning Studio.

### Download inference assets

### Audio inference

### Image inference

### Video inference

### Multimodal inference

## Release resources

Once you are done using the Foundry Endpoint, you can delete the resources (i.e., you will stop paying for the instance on which the model is running and all the attached costs) as follows:

In [ ]:
client.online_endpoints.begin_delete(name=os.getenv("ENDPOINT_NAME")).result()

## Conclusion

Throughout this example, you learned how to deploy `nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-FP8` as an Azure Machine Learning Managed Online Endpoint on Microsoft Foundry, and how to exercise its key inference capabilities: multimodal understanding of images, videos, and audio, as well as standard text-based inference.

If you have any doubt, issue or question about this example, feel free to [open an issue](https://github.com/huggingface/Microsoft-Azure/issues/new) and we'll do our best to help!